# FastWAM - LIBERO world-action model (interactive + imagination)

Standalone demo of **FastWAM** - a *world-action* model built on **Wan2.2-TI2V-5B** (a T5 text
encoder, a Wan video VAE, and a video+action diffusion transformer) - driving the **LIBERO**
simulator, rendered **inline, right here in the notebook**. Nothing to train and nothing to
download: the model weights and a few ground-truth episodes are already baked into this image.

Two things happen below, both on the **fast route** our AMD Strix Halo port ships:

1. **Interactive simulator (fast action route).** Exactly like the MolmoAct2 interactive demo, but
   driven by FastWAM. From the current camera view + robot state + your instruction, the model
   *plans a short action chunk* (`infer_action`), the simulator executes it, then it re-plans - the
   same receding-horizon closed loop as the shipped LIBERO eval. Type an instruction, press
   **Send**, and watch the arm act; the status line shows the live per-plan latency.

2. **Video imagination - what the model \"dreams\".** FastWAM is a *world* model, so on its joint
   route (`infer_joint`) it can roll the future **video** forward as well as the actions. We imagine
   **two full episodes from two different tasks** as receding-horizon rollouts and show **ground
   truth on the left, FastWAM's imagined future on the right** so you can compare.

Same self-contained style as the other notebooks: the sim's small web UI is served inside your
session and proxied through your JupyterHub route (`jupyter-server-proxy`), so there is **no extra
port to forward**. FastWAM runs in its own isolated environment (`/opt/fastwam-venv`), kept separate
from the MolmoAct2 training environment.

**Baked assets:** BF16 checkpoint + Wan2.2 base under `/opt/fastwam-assets` (resolved automatically
below). **Knobs (env):** `SUITE=libero_object`, `TASK_ID=0`, `EPISODES=47,273` (the two imagined
episodes), `NUM_STEPS=20`, `RT_PORT=8080`.

In [ ]:
import os
import subprocess
import json
import time
import urllib.request

# Quiet ROCm/torch spam in any child process we spawn.
os.environ.setdefault("TORCH_BLAS_PREFER_HIPBLASLT", "0")
os.environ.setdefault("PYTHONWARNINGS", "ignore")

OUT_DIR = os.environ.get("OUT_DIR", "/home/jovyan/outputs")
os.makedirs(OUT_DIR, exist_ok=True)

# FastWAM runs in its OWN isolated venv (the numpy-1.26.4 LIBERO sim stack + ROCm torch), separate
# from the MolmoAct2 train-venv, and on its OWN LIBERO config path so the two never clash.
FASTWAM_PY = os.path.join(os.environ.get("FASTWAM_VENV", "/opt/fastwam-venv"), "bin", "python")
LIBERO_CONFIG_PATH_FASTWAM = os.environ.get("LIBERO_CONFIG_PATH_FASTWAM", "/opt/libero-config-fastwam")

# Baked weight/data locations (set in the image; overridable via env). The checkpoint and the
# Wan2.2 base are already BF16 - the production precision the fast route runs in.
RELEASE_DIR = os.environ.get("FASTWAM_RELEASE_DIR", "/opt/fastwam-assets/fastwam_release")
DATA_DIR = os.environ.get("FASTWAM_DATA_DIR", "/opt/fastwam-assets/data")
DIFFSYNTH = os.environ.get("DIFFSYNTH_MODEL_BASE_PATH", "/opt/fastwam-assets/diffsynth")
CKPT = os.environ.get("CKPT") or os.path.join(RELEASE_DIR, "libero_uncond_2cam224.pt")
DATASET_STATS = os.environ.get("DATASET_STATS") or os.path.join(
    RELEASE_DIR, "libero_uncond_2cam224_dataset_stats.json")


# Subprocesses (the sim server / videogen) get a clean env: drop the notebook's inline matplotlib
# backend (crashes headless children), force plain hipBLAS + unbuffered output, run against the
# fastwam venv's LIBERO config, and select the FastWAM policy for the model-agnostic sim harness.
def child_env(**extra):
    e = dict(os.environ)
    e.pop("MPLBACKEND", None)
    e.update({
        "TORCH_BLAS_PREFER_HIPBLASLT": "0", "PYTHONWARNINGS": "ignore", "PYTHONUNBUFFERED": "1",
        "LIBERO_CONFIG_PATH": LIBERO_CONFIG_PATH_FASTWAM,
        "POLICY_FACTORY": "fastwam_libero_policy:build_policy",
        "CKPT": CKPT, "DATASET_STATS": DATASET_STATS,
    })
    e.update({k: str(v) for k, v in extra.items()})
    return e


# Fail early with a clear message if the assets aren't present (e.g. a code-only image without the
# baked weights). In the workshop image these are all baked in under /opt/fastwam-assets.
_missing = [p for p in (CKPT, DATASET_STATS, DIFFSYNTH) if not os.path.exists(p)]
if _missing:
    raise FileNotFoundError(
        "FastWAM assets not found:\n  " + "\n  ".join(_missing) +
        "\nThis notebook expects the with-assets workshop image (weights baked under "
        "/opt/fastwam-assets). Organizers: build with the fastwam bundle (see ORGANIZER.md)."
    )
print("FastWAM assets OK")
print("  checkpoint    :", CKPT)
print("  dataset stats :", DATASET_STATS)
print("  Wan2.2 base   :", DIFFSYNTH)
print("  GT episodes   :", DATA_DIR)
print("  interpreter   :", FASTWAM_PY)

## 1. Interactive LIBERO simulator (fast action route)

Run the cell below, wait for the model to load (the first run JIT-compiles GPU kernels - it can take
a couple of minutes), then the live sim appears inline. Pick a task from the **environment**
dropdown, type an instruction, and press **Send**. The status line shows the step count and the
live **per-plan latency** (the fast `infer_action` time). This is the default, fastest route.

In [ ]:
from IPython.display import IFrame, display

# Internal port inside your pod; reached only through the authenticated JupyterHub proxy (never
# exposed directly). The server binds 0.0.0.0:RT_PORT; jupyter-server-proxy forwards
# {JUPYTERHUB_SERVICE_PREFIX}/proxy/RT_PORT/ -> 127.0.0.1:RT_PORT.
RT_PORT = os.environ.get("RT_PORT", "8080")
SIM_SERVER = "/ryzers/notebooks/scripts/interactive_server_fastwam.py"

# Stop a server started by a previous run of this cell.
try:
    if globals().get("_sim") and _sim.poll() is None:
        _sim.terminate()
        _sim.wait(timeout=5)
except Exception:
    pass

# Serve the FastWAM policy on the fast action route. Stream server logs to a file so the notebook
# kernel never blocks on a full stdout pipe while the sim runs.
sim_env = child_env(
    PORT=RT_PORT,
    SUITE=os.environ.get("SUITE", "libero_object"),
    TASK_ID=os.environ.get("TASK_ID", "0"),
)
_log_path = os.path.join(OUT_DIR, "interactive_fastwam.log")
_log = open(_log_path, "w")
_sim = subprocess.Popen([FASTWAM_PY, SIM_SERVER], env=sim_env, stdout=_log, stderr=subprocess.STDOUT)
print(f"FastWAM sim server started (pid {_sim.pid}); loading T5 + VAE + DiT ...")
print("(first run JIT-compiles ROCm kernels; this can take a few minutes)")


def _sim_status():
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{RT_PORT}/status", timeout=3) as r:
            return json.load(r)
    except Exception:
        return None


ready, deadline = False, time.time() + 1200
while time.time() < deadline:
    if _sim.poll() is not None:
        print("\nserver exited early; tail of log:")
        print("".join(open(_log_path).readlines()[-25:]))
        break
    s = _sim_status()
    if s:
        print("  " + str(s.get("status", ""))[:96], end="\r")
        if s.get("mode") in ("idle", "running"):
            ready = True
            break
        if s.get("mode") == "error":
            print("\nserver error:", s.get("status"))
            print("".join(open(_log_path).readlines()[-25:]))
            break
    time.sleep(3)

prefix = os.environ.get("JUPYTERHUB_SERVICE_PREFIX", "").rstrip("/")
sim_url = f"{prefix}/proxy/{RT_PORT}/" if prefix else f"http://localhost:{RT_PORT}/"
if ready:
    print(f"\nready - interactive FastWAM sim embedded below (also open in a tab: {sim_url})")
    display(IFrame(sim_url, width="100%", height=860))
else:
    print(f"\nserver not ready yet; wait a moment and re-run this cell. URL: {sim_url}")

## 2. Video imagination - what the model "dreams"

FastWAM is a **world** model, so it can imagine the *future video*, not just the next actions. The
cell below imagines **two full episodes from two different LIBERO-Object tasks** - the *bbq sauce*
pick and the *milk* pick - and shows **ground truth (left) vs FastWAM's imagination (right)** side
by side, for the entire episode.

To cover a whole episode (not just one 32-step chunk) it runs `infer_joint` as a **receding-horizon
rollout**: each call imagines the next horizon of video+actions from the real observation at that
step, and the segments are stitched into one full-length clip - the same receding-horizon loop the
shipped LIBERO eval uses.

This is the heavier route (a full video diffusion per horizon), so the cell first **stops the live
sim above** to free the GPU, and the **first horizon JIT-compiles ROCm kernels** (a few minutes;
each subsequent horizon is far faster). Knobs: `EPISODES="47,273"` picks the two episodes (any
episode indices; different tasks live at different indices), `NUM_STEPS` trades speed for quality,
`MAX_STEPS_PER_EPISODE` caps length if you want a quicker preview.

In [ ]:
from IPython.display import Video, HTML, display

# Free the GPU: stop the interactive sim (both would otherwise hold a full model copy in memory).
try:
    if globals().get("_sim") and _sim.poll() is None:
        print("stopping the live sim to free the GPU for imagination ...")
        _sim.terminate()
        _sim.wait(timeout=10)
except Exception:
    pass

VIDEOGEN = "/ryzers/notebooks/scripts/fastwam_videogen.py"
# Two DIFFERENT tasks, rendered as FULL episodes (receding-horizon rollouts, not a single chunk):
# ep47 = "pick up the bbq sauce...", ep273 = "pick up the milk...". Override via EPISODES="e0,e1".
gen_env = child_env(
    EPISODES=os.environ.get("EPISODES", "47,273"),
    NUM_STEPS=os.environ.get("NUM_STEPS", "20"),
    TAG=os.environ.get("SUITE", "libero_object"),
)
print("imagining FULL episodes for two tasks (loads the model, then runs the joint video+action")
print("route as a receding-horizon rollout over each whole episode)...")
print("the first horizon JIT-warms ROCm kernels, so this takes a few minutes.\n")

# Stream the backend's output live and pick up the machine-parseable lines it prints per clip.
proc = subprocess.Popen([FASTWAM_PY, VIDEOGEN], env=gen_env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
mp4s, summary = [], None
for line in proc.stdout:
    line = line.rstrip()
    if line.startswith("VIDEOGEN_MP4="):
        mp4s.append(line.split("=", 1)[1])
    elif line.startswith("VIDEOGEN_SUMMARY="):
        try:
            summary = json.loads(line.split("=", 1)[1])
        except Exception:
            pass
    else:
        print(line)
proc.wait()

clip_task = {c["mp4"]: c for c in (summary.get("clips", []) if summary else [])}
if summary:
    print(f"\njoint-path latency ({summary['num_inference_steps']} steps): first horizon "
          f"{summary['joint_latency_s_first_warmup']:.1f}s (JIT warmup), then "
          f"~{summary['joint_latency_s_mean_steady']:.1f}s per horizon")
for i, mp4 in enumerate(mp4s):
    if os.path.exists(mp4):
        c = clip_task.get(mp4, {})
        task = c.get("task", f"clip {i}")
        meta = f" &mdash; {c['frames']} frames, {c['windows']} horizons" if c else ""
        display(HTML(f"<b>Full episode &mdash; {task}</b><br>"
                     f"<span>ground truth (left) vs FastWAM imagined (right){meta}</span>"))
        display(Video(mp4, embed=True, width=900))
    else:
        print("missing clip:", mp4)
if not mp4s:
    print("no clips produced - check the log above.")